#  HR Policy Assistant — QLoRA Fine-Tuning v2
### Model: `TinyLlama-1.1B-Chat` | Dataset: `hr_policy_dataset.json` (Heavy)

> **Project**: Domain-Specific HR Policy Assistant for Sri Lankan Companies  
> **Author**: Chamindu Nipun  
> **Version**: v2 — Full dataset, all API fixes applied, improved inference

---


## Step 1: Check GPU

In [ ]:
!nvidia-smi

## Step 2: Install Dependencies

In [ ]:
!pip install -q -U transformers>=4.41.0
!pip install -q -U peft>=0.10.0
!pip install -q -U trl>=0.8.6
!pip install -q -U bitsandbytes>=0.41.3
!pip install -q -U accelerate>=0.29.3
!pip install -q -U datasets>=2.19.0
!pip install -q evaluate rouge_score

print("All packages installed! Now RESTART RUNTIME before continuing.")

## Step 3: Verify Imports
**Restart runtime after install**, then run from this cell.

In [ ]:
import json
import torch
import warnings
warnings.filterwarnings('ignore')

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer

import transformers, peft, trl, bitsandbytes, accelerate, datasets

print("All imports successful!")
print(f"   torch          : {torch.__version__}")
print(f"   transformers   : {transformers.__version__}")
print(f"   peft           : {peft.__version__}")
print(f"   trl            : {trl.__version__}")
print(f"   bitsandbytes   : {bitsandbytes.__version__}")
print(f"   accelerate     : {accelerate.__version__}")
print(f"   datasets       : {datasets.__version__}")
print(f"   CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU            : {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 4: Upload & Load Dataset

Upload your `hr_policy_dataset.json` file when prompted.

In [ ]:
# Upload hr_policy_dataset.json
from google.colab import files

print("Please upload your hr_policy_dataset.json file.")
uploaded = files.upload()

# Load the uploaded JSON dataset
with open('hr_policy_dataset.json', 'r', encoding='utf-8') as f:
    HR_DATA = json.load(f)

print(f"\n Dataset loaded: {len(HR_DATA)} Q&A pairs")
print(f"\n Sample entry preview:")
sample = HR_DATA[0]
for msg in sample['messages']:
    print(f"  [{msg['role'].upper()}]: {msg['content'][:80]}...")

## Step 5: Validate & Prepare Dataset

In [ ]:
# Validate dataset structure
def validate_entry(entry, idx):
    """Check each entry has correct messages format."""
    assert 'messages' in entry, f"Entry {idx} missing 'messages' key"
    msgs = entry['messages']
    roles = [m['role'] for m in msgs]
    assert 'system' in roles,    f"Entry {idx} missing 'system' role"
    assert 'user' in roles,      f"Entry {idx} missing 'user' role"
    assert 'assistant' in roles, f"Entry {idx} missing 'assistant' role"
    return True

errors = []
for i, entry in enumerate(HR_DATA):
    try:
        validate_entry(entry, i)
    except AssertionError as e:
        errors.append(str(e))

if errors:
    print(f" Found {len(errors)} invalid entries:")
    for e in errors[:5]:
        print(f"   {e}")
else:
    print(f" All {len(HR_DATA)} entries are valid!")

# Dataset statistics
answer_lengths = [len(e['messages'][2]['content'].split()) for e in HR_DATA]
print(f"\n📊 Dataset Statistics:")
print(f"   Total Q&A pairs  : {len(HR_DATA)}")
print(f"   Avg answer words : {sum(answer_lengths) // len(answer_lengths)}")
print(f"   Min answer words : {min(answer_lengths)}")
print(f"   Max answer words : {max(answer_lengths)}")

In [ ]:
# Format dataset into TinyLlama chat template
def format_chat_prompt(example):
    messages  = example["messages"]
    formatted = ""
    for msg in messages:
        role    = msg["role"]
        content = msg["content"]
        if role == "system":
            formatted += f"<|system|>\n{content}</s>\n"
        elif role == "user":
            formatted += f"<|user|>\n{content}</s>\n"
        elif role == "assistant":
            formatted += f"<|assistant|>\n{content}</s>\n"
    return {"text": formatted}

# Convert + format
raw_dataset       = Dataset.from_list(HR_DATA)
formatted_dataset = raw_dataset.map(format_chat_prompt)

# Train / Val / Test  split  80% / 10% / 10%
split_1       = formatted_dataset.train_test_split(test_size=0.2, seed=42)
split_2       = split_1["test"].train_test_split(test_size=0.5, seed=42)
train_dataset = split_1["train"]
val_dataset   = split_2["train"]
test_dataset  = split_2["test"]

print(f"Train samples : {len(train_dataset)}")
print(f"Val samples   : {len(val_dataset)}")
print(f"Test samples  : {len(test_dataset)}")
print(f"\nSample formatted text:")
print(train_dataset[0]["text"])

## Step 6: Load Clean Base Model (4-bit NF4 Quantization)


In [ ]:
# Model choice
# Free Colab T4  → TinyLlama-1.1B  (~4GB VRAM, same LLaMA-2 architecture)
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit NF4 Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Tokenizer
print(f"Loading tokenizer from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded")

# Clean Base Model
print("Loading clean base model in 4-bit NF4 quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache      = False
model.config.pretraining_tp = 1
print("Clean base model loaded — LoRA will be applied by SFTTrainer")

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {used:.2f} GB used / {total:.1f} GB total")

## Step 7: Configure LoRA Adapters

In [ ]:
# LoRA Config (passed to SFTTrainer — NOT applied manually)
lora_config = LoraConfig(
    r=16,                                    # LoRA rank
    lora_alpha=32,                           # Scaling (2x rank = standard)
    target_modules=["q_proj", "v_proj"],     # Query & value projections
    lora_dropout=0.05,                       # Reduced dropout for larger dataset
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("LoRA config created (SFTTrainer will inject adapters)")
print(f"   Rank (r)       : {lora_config.r}")
print(f"   Alpha          : {lora_config.lora_alpha}")
print(f"   Target modules : {lora_config.target_modules}")
print(f"   Dropout        : {lora_config.lora_dropout}")

## Step 8: Configure SFTConfig


In [ ]:
OUTPUT_DIR = "./hr-policy-assistant"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Dataset
    dataset_text_field="text",           # confirmed in SFTConfig
    max_length=512,                      # confirmed in SFTConfig (NOT max_seq_length)
    packing=False,

    # Training Duration
    num_train_epochs=5,                  # 5 epochs for larger dataset
    max_steps=-1,

    # Batch & Gradient
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,       # Effective batch = 2 x 4 = 8

    # Optimizer & LR
    optim="paged_adamw_8bit",            # Memory-efficient for QLoRA
    learning_rate=1e-4,                  # Reduced LR for stable training
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_steps=20,                     # Increased warmup for larger dataset

    # Precision
    fp16=False,
    bf16=True,

    # Evaluation & Saving
    eval_strategy="epoch",               # confirmed in SFTConfig
    save_strategy="epoch",
    save_total_limit=2,                  # Keep only best 2 checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_steps=10,
    report_to="none",
)

print(" SFTConfig configured successfully!")
print(f"   Epochs         : {sft_config.num_train_epochs}")
print(f"   Learning rate  : {sft_config.learning_rate}")
print(f"   Batch size     : {sft_config.per_device_train_batch_size} x {sft_config.gradient_accumulation_steps} = {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps} effective")

## Step 9: Initialize SFTTrainer and Train

In [ ]:
# SFTTrainer
# Clean base model + peft_config → SFTTrainer applies LoRA internally
# processing_class replaces deprecated 'tokenizer' param (trl 0.9+)

trainer = SFTTrainer(
    model=model,                         # clean base model (no LoRA yet)
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,             # SFTTrainer injects LoRA internally
    processing_class=tokenizer,          # replaces deprecated 'tokenizer' kwarg
    args=sft_config,
)

print("Starting QLoRA fine-tuning...")
print(f"   Model      : {MODEL_NAME}")
print(f"   Train size : {len(train_dataset)} samples")
print(f"   Val size   : {len(val_dataset)} samples")
print(f"   Epochs     : {sft_config.num_train_epochs}")
print("-" * 50)

trainer.train()

print("\n Training complete!")

# Show trainable parameters
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in trainer.model.parameters())
print(f" Trainable params : {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")

## Step 10: Plot Training & Validation Loss

In [ ]:
import matplotlib.pyplot as plt

log_history  = trainer.state.log_history
train_losses = [(x["step"], x["loss"])      for x in log_history if "loss" in x]
eval_losses  = [(x["step"], x["eval_loss"]) for x in log_history if "eval_loss" in x]

if train_losses:
    steps_tr, losses_tr = zip(*train_losses)
    plt.figure(figsize=(10, 4))
    plt.plot(steps_tr, losses_tr, label="Train Loss", color="steelblue")
    if eval_losses:
        steps_ev, losses_ev = zip(*eval_losses)
        plt.plot(steps_ev, losses_ev, label="Val Loss", color="darkorange", marker="o")
    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    plt.title("QLoRA Fine-Tuning — Training & Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("training_loss.png", dpi=150)
    plt.show()
    print(" Loss curve saved as training_loss.png")

# Print final losses
if eval_losses:
    print(f"\n Final Train Loss : {losses_tr[-1]:.4f}")
    print(f"Final Val Loss   : {losses_ev[-1]:.4f}")

## Step 11: Merge LoRA Adapters into Base Model & Save

In [ ]:
MERGED_MODEL_DIR = "./hr-policy-assistant-merged"

print(" Merging LoRA adapters into base model...")
merged_model = trainer.model.merge_and_unload()

merged_model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

print(f" Merged model saved to : {MERGED_MODEL_DIR}")
print("   Files: model.safetensors + tokenizer config")

# Check saved files
import os
files_saved = os.listdir(MERGED_MODEL_DIR)
print(f"   Saved files: {files_saved}")

## Step 12: Evaluate — ROUGE Score on Test Set

In [ ]:
import evaluate

rouge = evaluate.load("rouge")

#  Fixed generate_answer (no max_new_tokens + max_length conflict)
def generate_answer(question, max_new_tokens=250):
    """
    Generate an answer using the fine-tuned merged model.
    Uses .generate() directly to avoid max_new_tokens/max_length conflict.
    Decodes only newly generated tokens (not the prompt).
    """
    prompt = (
        "<|system|>\n"
        "You are an HR Policy Assistant for Sri Lankan companies. "
        "Answer questions accurately based on Sri Lankan labor laws and company HR policies "
        "in a formal, professional tone.</s>\n"
        f"<|user|>\n{question}</s>\n"
        "<|assistant|>\n"
    )

    inputs   = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = merged_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,   # only max_new_tokens — no conflict
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens (skip prompt)
    new_tokens = output_ids[0][input_len:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response


# Run ROUGE evaluation on held-out test set
print(f"Running ROUGE evaluation on {len(test_dataset)} test samples...")
predictions, references = [], []

for sample in test_dataset:
    messages  = sample["messages"]
    question  = messages[1]["content"]
    reference = messages[2]["content"]
    predictions.append(generate_answer(question))
    references.append(reference)

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print("\n ROUGE Evaluation Results (Test Set):")
print("-" * 45)
for metric, score in rouge_scores.items():
    print(f"  {metric.upper():<12} : {score:.4f}")

print("\n Sample prediction:")
q = test_dataset[0]['messages'][1]['content']
print(f"Q : {q}")
print(f"A : {predictions[0][:400]}")

## Step 13: Baseline Comparison
Compare fine-tuned model vs base model (no fine-tuning) on the same questions.

In [ ]:
#  Load the original base model for comparison
print(" Loading base model for baseline comparison...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print(" Base model loaded")

def generate_base_answer(question, max_new_tokens=200):
    prompt = (
        "<|system|>\n"
        "You are an HR Policy Assistant for Sri Lankan companies. "
        "Answer questions accurately based on Sri Lankan labor laws and company HR policies "
        "in a formal, professional tone.</s>\n"
        f"<|user|>\n{question}</s>\n"
        "<|assistant|>\n"
    )
    inputs    = tokenizer(prompt, return_tensors="pt").to(base_model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


compare_questions = [
    "What is the maternity leave entitlement in Sri Lanka?",
    "How many casual leaves can I take per year?",
    "What is the EPF contribution percentage?",
]

print("\n" + "=" * 70)
print("  BASELINE vs FINE-TUNED COMPARISON")
print("=" * 70)

for q in compare_questions:
    base_ans  = generate_base_answer(q)
    tuned_ans = generate_answer(q)
    print(f"\n {q}")
    print(f"\n BASE MODEL:")
    print(f"   {base_ans[:250]}")
    print(f"\n FINE-TUNED:")
    print(f"   {tuned_ans[:250]}")
    print("-" * 70)

## Step 14: Interactive Inference — Test Your HR Assistant

In [ ]:
test_questions = [
    "What is the maternity leave entitlement in Sri Lanka?",
    "How many casual leaves can I take per year?",
    "What is the EPF contribution percentage?",
    "Can I carry forward unused annual leave?",
    "What is the notice period if I resign?",
    "What is the gratuity entitlement?",
    "How is overtime calculated?",
    "What is the probation period for new employees?"
]

print("=" * 65)
print("    HR POLICY ASSISTANT — Interactive Test (Fine-Tuned Model)")
print("=" * 65)

for i, question in enumerate(test_questions, 1):
    print(f"\n[Q{i}] {question}")
    print("-" * 55)
    answer = generate_answer(question)
    print(f"[A]  {answer}")
    print()

## Step 15: Save to Google Drive (Optional)

In [ ]:
# Uncomment to save fine-tuned model to Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

# import shutil
# shutil.copytree(
#     MERGED_MODEL_DIR,
#     "/content/drive/MyDrive/hr-policy-assistant-merged"
# )
# print(" Model saved to Google Drive!")


## Step 16: Upload to Hugging Face Hub (Optional)

In [ ]:
# Uncomment and add HF token to upload

# from huggingface_hub import login
# login(token="hf_TOKEN_HERE")

# merged_model.push_to_hub("your-username/hr-policy-assistant-tinyllama")
# tokenizer.push_to_hub("your-username/hr-policy-assistant-tinyllama")
# print(" Model uploaded to Hugging Face Hub!")


---
## Pipeline Summary

| Stage | Details |
|---|---|
| Dataset | `hr_policy_dataset.json` (your full dataset) |
| Split | 80% train / 10% val / 10% test |
| Model | TinyLlama-1.1B-Chat (4-bit NF4 QLoRA) |
| LoRA | r=16, α=32, q_proj + v_proj, dropout=0.05 |
| Training | 5 epochs, lr=1e-4, paged_adamw_8bit, cosine LR |
| Evaluation | ROUGE-1/2/L on held-out test set + baseline comparison |
| Inference | `.generate()` directly — no max_new_tokens conflict |
| Save | Merged safetensors + optional Google Drive / HF Hub |
